In [9]:
import numpy as np
import pandas as pd

Import the novelists data extracted from Wikipedia and Fantastic Fiction.

In [10]:
novelists = pd.read_pickle('data/novelists.pkl')

For each novelist, create a column representing the total number of English works they published before winning the prize and the number of works they published in the 10-year period and 5-year period before they won the prize.

In [11]:
total_works = []
works_10_year = []
works_5_year = []
for i in novelists.index:
    year_won = novelists.loc[i,"Year"]
    years_of_publication = novelists.loc[i,"Works"]["Year"]
    total_works.append((years_of_publication < year_won).sum())
    works_10_year.append(((years_of_publication < year_won) & (years_of_publication >= (year_won-10))).sum())
    works_5_year.append(((years_of_publication < year_won) & (years_of_publication >= (year_won-5))).sum())


novelists["Works before prize"] = pd.Series(total_works)
novelists["Works within 10 years"] = pd.Series(works_10_year)
novelists["Works within 5 years"] = pd.Series(works_5_year)

To organize the data for visualizations, we aggregate these productivity features using a sliding window with a size of 10.

In [12]:
# Calculate moving average of works before prize

novelists['MA_works_before_prize'] = novelists['Works before prize'].rolling(window=10).mean()
novelists['MA_works_10_year'] = novelists['Works within 10 years'].rolling(window=10).mean()
novelists['MA_works_5_year'] = novelists['Works within 5 years'].rolling(window=10).mean()


For each of the productivity features we just created, we also create a new DataFrame. The features in these tables are:
- Author Name
- Book Title
- Publication Year

These tables can be merged onto the main novelists table if we need the information for each publication.

In [13]:
total_works = pd.DataFrame({"Name":[], "Title": [], "Publication Year": []})
for i in novelists.index:
    author = novelists.iloc[i]
    worksdf = author["Works"].rename({"Year":"Publication Year"}, axis=1)
    worksdf = worksdf[(worksdf["Publication Year"] < author["Year"])]
    worksdf["Name"] = author["Name"]
    total_works = pd.concat([total_works, worksdf], axis=0)

In [14]:
ten_year_works = pd.DataFrame({"Name":[], "Title": [], "Publication Year": []})
for i in novelists.index:
    author = novelists.iloc[i]
    worksdf = author["Works"].rename({"Year":"Publication Year"}, axis=1)
    worksdf = worksdf[(worksdf["Publication Year"] < author["Year"]) & (worksdf["Publication Year"] >= author["Year"]-10)]
    worksdf["Name"] = author["Name"]
    ten_year_works = pd.concat([ten_year_works, worksdf], axis=0)

In [15]:
five_year_works = pd.DataFrame({"Name":[], "Title": [], "Publication Year": []})
for i in novelists.index:
    author = novelists.iloc[i]
    worksdf = author["Works"].rename({"Year":"Publication Year"}, axis=1)
    worksdf = worksdf[(worksdf["Publication Year"] < author["Year"]) & (worksdf["Publication Year"] >= author["Year"]-5)]
    worksdf["Name"] = author["Name"]
    five_year_works = pd.concat([five_year_works, worksdf], axis=0)
    

In [16]:
ten_year_works.head()

,Name,Title,Publication Year
6,Henryk Sienkiewicz,Quo Vadis?,1896
7,Henryk Sienkiewicz,So Runs The World,1898
8,Henryk Sienkiewicz,The Knights of the Cross,1900
4,Rudyard Kipling,Puck of Pook's Hill,1906
8,Rudyard Kipling,Captains Courageous,1897


Write the prepared novelists data to a pickle file, and create csv files for the other DataFrames created.

In [17]:
novelists.to_pickle('data/prepared_novelists.pkl')
five_year_works.to_csv('data/five_year_works.csv')
ten_year_works.to_csv('data/ten_year_works.csv')
total_works.to_csv('data/total_works.csv')